# 06 — Optimización de hiperparámetros y selección de variables

Objetivo: ajustar hiperparámetros de los modelos candidatos y evaluar configuraciones compactas con validación cruzada.

## Carga de datos y preparación

Este bloque reproduce la división usada en los experimentos. Ejecutá el notebook desde la raíz del proyecto para que la ruta relativa funcione.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

df_fe = pd.read_csv("../data/processed/airbnb_dp.csv")

X = df_fe.drop(columns=["price"])
y = df_fe["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)


X_train: (19140, 136)
X_test: (4786, 136)


## 1. Optimización de hiperparámetros

Se realizan búsquedas sobre los candidatos prometedores del notebook 05. La evaluación principal para seleccionar hiperparámetros debe ser la validación cruzada; el test queda para una evaluación final.

## Ajustar los mejores modelos

In [ ]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import randint, loguniform, uniform


def model_pipeline_trees(X_train, y_train, X_test, y_test):

    models = {
        "Random Forest": {
            "model": RandomForestRegressor(random_state=42, n_jobs=1),
            "params": {
                "model__n_estimators": randint(100, 301),
                "model__max_depth": [10, 20, 30, None],
                "model__min_samples_split": randint(2, 16),
                "model__min_samples_leaf": randint(1, 11),
                "model__max_features": uniform(0.5, 0.5),
            },
            "n_iter": 20,
        },
        "Hist Gradient Boosting": {
            "model": HistGradientBoostingRegressor(random_state=42),
            "params": {
                "model__max_iter": randint(100, 301),
                "model__learning_rate": loguniform(0.02, 0.15),
                "model__max_leaf_nodes": randint(10, 64),
                "model__min_samples_leaf": randint(10, 41),
                "model__l2_regularization": loguniform(0.01, 10),
            },
            "n_iter": 20,
        },
    }

    results = {}

    for name, info in models.items():
        print(f"\nEntrenando: {name}")

        pipeline = Pipeline([("model", info["model"])])

        search = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=info["params"],
            n_iter=info["n_iter"],
            cv=3,
            scoring="r2",
            n_jobs=-1,
            random_state=42,
            verbose=2,
            return_train_score=True,
        )

        search.fit(X_train, y_train)

        best_model = search.best_estimator_
        predictions = best_model.predict(X_test)

        results[name] = {
            "Best Params": search.best_params_,
            "CV R2": search.best_score_,
            "Test R2": r2_score(y_test, predictions),
            "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
            "MAE": mean_absolute_error(y_test, predictions),
        }

        print("\nMejores parámetros:", search.best_params_)
        print("CV R²:", search.best_score_)
        print("Test R²:", results[name]["Test R2"])
        print("RMSE:", results[name]["RMSE"])
        print("MAE:", results[name]["MAE"])

    return pd.DataFrame(results).T


resultados_trees = model_pipeline_trees(X_train, y_train, X_test, y_test)

display(resultados_trees)



Entrenando: Random Forest
Fitting 3 folds for each of 20 candidates, totalling 60 fits


In [ ]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Transformar el target
y_train_log = np.log1p(y_train)

# Entrenar el modelo
modelo_log = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_iter=200,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    random_state=42,
)

modelo_log.fit(X_train, y_train_log)

# Predecir y volver a la escala original
predicciones_log = modelo_log.predict(X_test)
predicciones_precio = np.expm1(predicciones_log)

# Evaluar en la escala original
print("R²:", r2_score(y_test, predicciones_precio))
print("RMSE:", np.sqrt(mean_squared_error(y_test, predicciones_precio)))
print("MAE:", mean_absolute_error(y_test, predicciones_precio))

## 2. Selección automática con SelectFromModel

Se compara un HistGradientBoosting con selección automática basada en ExtraTrees. El selector debe permanecer dentro del `Pipeline` para ajustarse dentro de cada fold y evitar fuga de información.

### Selección automática con `SelectFromModel`

`SelectFromModel` utiliza un estimador auxiliar (`ExtraTreesRegressor`) para asignar importancias y conservar las variables que superan un umbral. El selector y HistGradientBoosting se colocan dentro de un `Pipeline`; de esta forma, en cada fold de validación cruzada, el selector aprende únicamente con la partición de entrenamiento de ese fold.

En una prueba inicial de umbrales, `1.25*mean` seleccionó 30 variables, obtuvo CV R² ≈ 0,5221 y test R² ≈ 0,5025. Esta versión compacta no superó al modelo completo en test. Además, optimizar después los hiperparámetros sobre las variables seleccionadas dio CV R² ≈ 0,5237, pero test R² ≈ 0,4962; la validación cruzada y el test no siempre ordenan los candidatos de la misma forma.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Selector de variables + modelo predictivo
pipeline_fs = Pipeline(
    [
        (
            "selector",
            SelectFromModel(
                estimator=ExtraTreesRegressor(
                    n_estimators=100, random_state=42, n_jobs=1
                ),
                threshold="median",
            ),
        ),
        (
            "model",
            HistGradientBoostingRegressor(
                learning_rate=0.068,
                max_iter=140,
                max_leaf_nodes=38,
                min_samples_leaf=24,
                l2_regularization=7.87,
                random_state=42,
            ),
        ),
    ]
)

# Probar distintos criterios para seleccionar variables
param_grid_fs = {"selector__threshold": ["mean", "1.25*mean", "1.5*mean", "2*mean"]}

grid_fs = GridSearchCV(
    estimator=pipeline_fs,
    param_grid=param_grid_fs,
    cv=3,
    scoring="r2",
    n_jobs=-1,
    verbose=2,
)

# Entrenar exclusivamente con el conjunto de entrenamiento
grid_fs.fit(X_train, y_train)

print("Mejor umbral:", grid_fs.best_params_)
print("Mejor R² de validación:", grid_fs.best_score_)

# Evaluar el mejor pipeline en test
mejor_modelo_fs = grid_fs.best_estimator_
predicciones_fs = mejor_modelo_fs.predict(X_test)

print("R² test:", r2_score(y_test, predicciones_fs))
print("RMSE:", np.sqrt(mean_squared_error(y_test, predicciones_fs)))
print("MAE:", mean_absolute_error(y_test, predicciones_fs))


In [ ]:
mejor_selector = grid_fs.best_estimator_.named_steps["selector"]

features_seleccionadas = X_train.columns[mejor_selector.get_support()]

print("Mejor umbral:", grid_fs.best_params_)
print("R² CV:", grid_fs.best_score_)
print("Cantidad de variables:", len(features_seleccionadas))
print("Variables seleccionadas:", features_seleccionadas.tolist())


In [ ]:
# Obtener las variables elegidas por SelectFromModel
mejor_pipeline_fs = grid_fs.best_estimator_

mascara = mejor_pipeline_fs.named_steps["selector"].get_support()
features_30 = X_train.columns[mascara]

# Transformar entrenamiento y test con el selector ya ajustado
X_train_30 = mejor_pipeline_fs.named_steps["selector"].transform(X_train)
X_test_30 = mejor_pipeline_fs.named_steps["selector"].transform(X_test)

# Entrenar el modelo final con las 30 variables
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

modelo_30 = HistGradientBoostingRegressor(
    learning_rate=0.068,
    max_iter=140,
    max_leaf_nodes=38,
    min_samples_leaf=24,
    l2_regularization=7.87,
    random_state=42,
)

modelo_30.fit(X_train_30, y_train)
predicciones_30 = modelo_30.predict(X_test_30)

print("Cantidad de variables:", len(features_30))
print("R²:", r2_score(y_test, predicciones_30))
print("RMSE:", np.sqrt(mean_squared_error(y_test, predicciones_30)))
print("MAE:", mean_absolute_error(y_test, predicciones_30))


## 3. Optimización de HistGradientBoosting con variables seleccionadas

Estas búsquedas comparan hiperparámetros para la variante reducida. Registrá por separado el rendimiento de CV y la evaluación de test.

In [ ]:
import numpy as np
import pandas as pd

from scipy.stats import randint, loguniform

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Modelo base
modelo_hgb = HistGradientBoostingRegressor(random_state=42)

# Espacio de búsqueda
param_distributions = {
    "learning_rate": loguniform(0.02, 0.12),
    "max_iter": randint(100, 301),
    "max_leaf_nodes": randint(15, 51),
    "min_samples_leaf": randint(10, 41),
    "l2_regularization": loguniform(0.01, 20),
}

# Búsqueda aleatoria
random_search_30 = RandomizedSearchCV(
    estimator=modelo_hgb,
    param_distributions=param_distributions,
    n_iter=30,
    cv=3,
    scoring="r2",
    n_jobs=-1,
    random_state=42,
    verbose=2,
)

# Entrenar solo con las variables seleccionadas
random_search_30.fit(X_train_30, y_train)

print("Mejores parámetros:")
print(random_search_30.best_params_)

print("\nMejor R² de validación cruzada:")
print(random_search_30.best_score_)


In [ ]:
import numpy as np
import pandas as pd

from scipy.stats import randint, loguniform

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Modelo base
modelo_hgb = HistGradientBoostingRegressor(random_state=42)

# Espacio de búsqueda
param_distributions = {
    "learning_rate": loguniform(0.02, 0.12),
    "max_iter": randint(100, 301),
    "max_leaf_nodes": randint(15, 51),
    "min_samples_leaf": randint(10, 41),
    "l2_regularization": loguniform(0.01, 20),
}

# Búsqueda aleatoria
random_search_30 = RandomizedSearchCV(
    estimator=modelo_hgb,
    param_distributions=param_distributions,
    n_iter=30,
    cv=3,
    scoring="r2",
    n_jobs=-1,
    random_state=42,
    verbose=2,
)

# Entrenar solo con las variables seleccionadas
random_search_30.fit(X_train_30, y_train)

print("Mejores parámetros:")
print(random_search_30.best_params_)

print("\nMejor R² de validación cruzada:")
print(random_search_30.best_score_)


In [ ]:
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

mejor_hgb_30 = random_search_30.best_estimator_

predicciones_hgb_30 = mejor_hgb_30.predict(X_test_30)

print("Variables:", X_train_30.shape[1])
print("R²:", r2_score(y_test, predicciones_hgb_30))
print("RMSE:", np.sqrt(mean_squared_error(y_test, predicciones_hgb_30)))
print("MAE:", mean_absolute_error(y_test, predicciones_hgb_30))


## 4. Comparación rigurosa de umbrales de selección

El selector se ajusta dentro de cada fold. Los resultados permiten estudiar el compromiso entre cantidad de variables y rendimiento. El test no debe usarse para elegir umbrales.

## Selección de variables evaluada con validación cruzada

Para comparar el modelo completo con el modelo seleccionado de forma más justa, se usó `KFold(n_splits=3, shuffle=True, random_state=42)` y se mantuvo HistGradientBoosting con los mismos hiperparámetros. La selección basada en ExtraTrees se ajusta dentro de cada fold.

En la primera comparación, el umbral `1.25*mean` conservó 29 variables al ajustarse sobre el entrenamiento completo. El modelo completo obtuvo CV R² ≈ 0,5221 y test R² ≈ 0,5075; el reducido obtuvo CV R² ≈ 0,5200 y test R² ≈ 0,5044. La selección reduce complejidad, pero en esa configuración pierde algo de rendimiento.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

# Mismo esquema de validación para los dos experimentos
cv = KFold(n_splits=3, shuffle=True, random_state=42)

# Parámetros del mejor HistGradientBoosting con todas las variables
hgb_params = {
    "l2_regularization": 7.87,
    "learning_rate": 0.068,
    "max_iter": 140,
    "max_leaf_nodes": 38,
    "min_samples_leaf": 24,
    "random_state": 42,
}

# Selector: aprende la importancia SOLO en cada fold de entrenamiento
selector = SelectFromModel(
    estimator=ExtraTreesRegressor(
        n_estimators=100, min_samples_leaf=2, random_state=42, n_jobs=-1
    ),
    threshold="1.25*mean",
)

# Modelo completo: todas las variables
modelo_completo = Pipeline([("hgb", HistGradientBoostingRegressor(**hgb_params))])

# Modelo reducido: selección dentro de cada fold
modelo_seleccion = Pipeline(
    [("selector", selector), ("hgb", HistGradientBoostingRegressor(**hgb_params))]
)

modelos = {"Completo": modelo_completo, "Seleccion de variables": modelo_seleccion}

resultados = []

for nombre, modelo in modelos.items():
    scores = cross_validate(
        modelo,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "r2": "r2",
            "mae": "neg_mean_absolute_error",
            "rmse": "neg_root_mean_squared_error",
        },
        n_jobs=1,
        return_train_score=False,
    )

    resultados.append(
        {
            "Modelo": nombre,
            "CV R2": scores["test_r2"].mean(),
            "CV R2 std": scores["test_r2"].std(),
            "CV RMSE": -scores["test_rmse"].mean(),
            "CV MAE": -scores["test_mae"].mean(),
        }
    )

comparacion_cv = pd.DataFrame(resultados)
comparacion_cv.sort_values("CV R2", ascending=False)


In [ ]:
resultados_test = []

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    predicciones = modelo.predict(X_test)

    fila = {
        "Modelo": nombre,
        "Test R2": r2_score(y_test, predicciones),
        "Test RMSE": root_mean_squared_error(y_test, predicciones),
        "Test MAE": mean_absolute_error(y_test, predicciones),
    }

    if nombre == "Seleccion de variables":
        selector_ajustado = modelo.named_steps["selector"]
        fila["Variables seleccionadas"] = int(selector_ajustado.get_support().sum())
    else:
        fila["Variables seleccionadas"] = X_train.shape[1]

    resultados_test.append(fila)

comparacion_test = pd.DataFrame(resultados_test)
comparacion_test


## Búsqueda de un modelo compacto

Se evaluaron cinco umbrales de importancia manteniendo los mismos hiperparámetros de HistGradientBoosting. El selector se ajustó dentro de cada fold de CV. La tabla resume tanto el rendimiento promedio en CV como la cantidad de variables que el selector conservó al ajustarse sobre todo `X_train`.

| Configuración | Variables | CV R² | CV R² std | CV RMSE | CV MAE |
|---|---:|---:|---:|---:|---:|
| Completo | 136 | 0,5221 | 0,0073 | 60.612,38 | 34.555,90 |
| `0.5*mean` | 43 | **0,5232** | **0,0054** | **60.544,91** | **34.548,93** |
| `mean` | 31 | 0,5190 | 0,0077 | 60.802,80 | 34.669,85 |
| `1.25*mean` | 29 | 0,5200 | 0,0068 | 60.743,24 | 34.640,01 |
| `1.5*mean` | 24 | 0,5176 | 0,0088 | 60.897,35 | 34.792,88 |
| `2*mean` | 17 | 0,5101 | 0,0104 | 61.363,67 | 35.332,20 |

El umbral `0.5*mean` fue el mejor de las configuraciones compactas en CV y conservó 43 variables. La diferencia con el modelo completo es mínima y la desviación estándar de R² entre folds fue algo menor. Sin embargo, el resultado de CV por sí solo no determina qué modelo rendirá mejor en datos nuevos, así que también se comparó en test.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import KFold, cross_validate

cv = KFold(n_splits=3, shuffle=True, random_state=42)

hgb_params = {
    "l2_regularization": 7.87,
    "learning_rate": 0.068,
    "max_iter": 140,
    "max_leaf_nodes": 38,
    "min_samples_leaf": 24,
    "random_state": 42,
}

thresholds = ["0.5*mean", "mean", "1.25*mean", "1.5*mean", "2*mean"]

resultados_seleccion = []

# Referencia: modelo completo
modelos = {"Completo": Pipeline([("hgb", HistGradientBoostingRegressor(**hgb_params))])}

# Cada selector se ajusta dentro de cada fold
for threshold in thresholds:
    modelos[threshold] = Pipeline(
        [
            (
                "selector",
                SelectFromModel(
                    ExtraTreesRegressor(
                        n_estimators=100, min_samples_leaf=2, random_state=42, n_jobs=-1
                    ),
                    threshold=threshold,
                ),
            ),
            ("hgb", HistGradientBoostingRegressor(**hgb_params)),
        ]
    )

for nombre, modelo in modelos.items():
    scores = cross_validate(
        modelo,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "r2": "r2",
            "rmse": "neg_root_mean_squared_error",
            "mae": "neg_mean_absolute_error",
        },
        n_jobs=1,
    )

    resultados_seleccion.append(
        {
            "Configuracion": nombre,
            "CV R2": scores["test_r2"].mean(),
            "CV R2 std": scores["test_r2"].std(),
            "CV RMSE": -scores["test_rmse"].mean(),
            "CV MAE": -scores["test_mae"].mean(),
        }
    )

tabla_seleccion = pd.DataFrame(resultados_seleccion)
tabla_seleccion.sort_values("CV R2", ascending=False).round(4)


In [ ]:
conteo_variables = []

for threshold in thresholds:
    selector = SelectFromModel(
        ExtraTreesRegressor(
            n_estimators=100, min_samples_leaf=2, random_state=42, n_jobs=-1
        ),
        threshold=threshold,
    )

    selector.fit(X_train, y_train)

    conteo_variables.append(
        {
            "Umbral": threshold,
            "Variables seleccionadas": int(selector.get_support().sum()),
            "Variables originales": X_train.shape[1],
        }
    )

tabla_variables = pd.DataFrame(conteo_variables)
tabla_variables


### Comparación final: 136 frente a 43 variables

| Métrica | Completo (136) | Reducido (43) |
|---|---:|---:|
| CV R² | 0,5221 | **0,5232** |
| CV RMSE | 60.612,38 | **60.544,91** |
| CV MAE | 34.555,90 | **34.548,93** |
| Test R² | **0,5075** | 0,5061 |
| Test RMSE | **65.395,19** | 65.488,10 |
| Test MAE | **35.017,63** | 35.158,32 |
| Número de variables | 136 | **43** |

El modelo de 43 variables reduce el número de columnas en 93 (≈68,4 %). En test, su R² baja alrededor de 0,0014, el RMSE sube unos 92,92 (≈0,14 %) y el MAE sube unos 140,69 (≈0,40 %), respecto al modelo completo. Por lo tanto, ofrece una simplificación importante a cambio de una pérdida predictiva pequeña. No se debe describir como un modelo que mejora el test; su ventaja principal es ser más compacto.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import pandas as pd

hgb_params = {
    "l2_regularization": 7.87,
    "learning_rate": 0.068,
    "max_iter": 140,
    "max_leaf_nodes": 38,
    "min_samples_leaf": 24,
    "random_state": 42,
}

modelo_completo = HistGradientBoostingRegressor(**hgb_params)

modelo_43 = Pipeline(
    [
        (
            "selector",
            SelectFromModel(
                ExtraTreesRegressor(
                    n_estimators=100, min_samples_leaf=2, random_state=42, n_jobs=-1
                ),
                threshold="0.5*mean",
            ),
        ),
        ("hgb", HistGradientBoostingRegressor(**hgb_params)),
    ]
)

modelos_test = {
    "Completo (136 variables)": modelo_completo,
    "Reducido (umbral 0.5*mean)": modelo_43,
}

resultados_finales = []

for nombre, modelo in modelos_test.items():
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)

    n_vars = (
        modelo.named_steps["selector"].get_support().sum()
        if nombre.startswith("Reducido")
        else X_train.shape[1]
    )

    resultados_finales.append(
        {
            "Modelo": nombre,
            "Variables": int(n_vars),
            "Test R2": r2_score(y_test, pred),
            "Test RMSE": root_mean_squared_error(y_test, pred),
            "Test MAE": mean_absolute_error(y_test, pred),
        }
    )

pd.DataFrame(resultados_finales).round(4)


In [ ]:
selector_43 = modelo_43.named_steps["selector"]

variables_seleccionadas = X_train.columns[selector_43.get_support()].tolist()

print(f"Cantidad de variables: {len(variables_seleccionadas)}")

for variable in variables_seleccionadas:
    print(variable)

## Conclusiones y decisión

1. **Feature engineering:** añadir cinco variables derivadas produjo una mejora pequeña en las métricas de test de la comparación registrada. Es conveniente validar esa mejora mediante CV antes de considerarla definitiva.
2. **PCA:** no mejoró Ridge y prácticamente no cambió el rendimiento de SVR; se descarta como estrategia principal en este experimento.
3. **Selección de variables:** el umbral `0.5*mean` seleccionó 43 de 136 variables. En validación cruzada sus métricas fueron levemente mejores que las del modelo completo; en test quedó ligeramente por debajo.
4. **Candidato compacto:** el pipeline de 43 variables es una alternativa válida cuando importan la simplicidad y el menor número de características. El modelo completo continúa siendo la referencia con el mejor resultado de test de esta comparación.

### Variables retenidas en la versión compacta

**Características del alojamiento:** `superhost`, `accommodates`, `bedrooms`, `beds`, `bathrooms`, `bathroom_shared`, `bathroom_unspecified`, `min_nights`, `max_nights`, `availability_365`, `number_of_reviews`, `review_scores_rating`.

**Ubicación:** `latitude`, `longitude`, `neighbourhood_cleansed_Belgrano`, `neighbourhood_cleansed_Colegiales`, `neighbourhood_cleansed_Palermo`, `neighbourhood_cleansed_Puerto Madero`, `neighbourhood_cleansed_Recoleta`.

**Tipo de propiedad y habitación:** `property_type_Entire condo`, `property_type_Entire home`, `property_type_Entire loft`, `property_type_Entire rental unit`, `room_type_Entire home/apt`.

**Amenities:** `Wifi`, `Hot water`, `Dishes and silverware`, `Cooking basics`, `Hangers`, `Bed linens`, `Hair dryer`, `Refrigerator`, `Microwave`, `Bidet`, `Air conditioning`, `Essentials`, `Hot water kettle`, `Elevator`, `Room-darkening shades`, `TV`, `Dedicated workspace`, `Iron`, `Extra pillows and blankets`.

### Limitación metodológica

El conjunto de test se consultó durante varios pasos de experimentación y selección. Para una evaluación final más rigurosa, conviene congelar las decisiones del modelo y utilizar un test independiente no usado para decidir entre variantes, o aplicar una estrategia de validación anidada. El umbral y la selección de variables deben formar parte del `Pipeline` que se ajuste solamente con datos de entrenamiento.


## Cierre del notebook 06

Seleccioná el modelo y los hiperparámetros basándote en validación cruzada. La evaluación final y el análisis detallado de errores se documentan en el notebook 07.